In [2]:
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from tqdm import tqdm
from datasets import Dataset
import torch

c:\Users\skhan\Documents\GitHub\NLP_HW2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
device = torch.device("mps") if torch.backends.mps.is_available() else \
        torch.device("cuda") if torch.cuda.is_available() else \
        torch.device("cpu")

In [3]:
print(f"Using device: {device}")

Using device: cpu


# Loading the MADLAD Translation Model

In [4]:
# Define model name
model_name = "google/madlad400-3b-mt"
model = AutoModelForSeq2SeqLM.from_pretrained(model_name, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Reading the Dataset

We'll load the CSV file containing Old Italian sentences that need to be translated to Modern Italian

In [17]:
# Load dataset
df = pd.read_csv("dataset.csv")

# Display basic information
print("DataFrame shape:", df.shape)
print("\nFirst 5 rows:")
df.head()


DataFrame shape: (97, 4)

First 5 rows:


,Author,Date,Region,Sentence
0,Brunetto Latini,1260-61,fior.,quella guerra ben fatta l' opera perché etc. E...
1,Bono Giamboni,1292,fior.,"crudele, e di tutte le colpe pigli vendetta, c..."
2,Valerio Massimo (red. V1,1336,fior.,Non d' altra forza d' animo fue ornato Ponzio ...
3,Lucano volg. (ed. Marinoni),1330/40,prat.,Se questo piace a tutti e se 'l tempo hae biso...
4,Brunetto Latini,1260-61,fior.,Officio di questa arte pare che sia dicere app...


# Setting up Translator 

In [ ]:
df["Sentence"] = df["Sentence"].apply(lambda x: "<2it> " + x)

# Translating the Sentences

We'll now translate all sentences in the dataset from Old Italian to Modern Italian and add them as a new column.

In [ ]:
translations = []

for i in tqdm(range(len(df))):
    input_text = "<2it> " + df.iloc[i]["Sentence"]
    input_ids = tokenizer(input_text, return_tensors="pt").input_ids.to(model.device)
    outputs = model.generate(input_ids=input_ids)
    translation = tokenizer.decode(outputs[0], skip_special_tokens=True)
    translations.append(translation)

# Add translations to the test dataframe
df['Modern_Italian'] = translations

# Show the results
print("\nTranslation Results:")
df[['Sentence', 'Modern_Italian']].head(10)

100%|██████████| 97/97 [12:21<00:00,  7.64s/it]


Translation Results:


,Sentence,Modern_Italian
0,<2it> quella guerra ben fatta l' opera perché ...,Et dall' altra parte Aiaces era un cavaliere f...
1,"<2it> crudele, e di tutte le colpe pigli vende...","E di tutte le colpe pigli vendetta, come dice ..."
2,<2it> Non d' altra forza d' animo fue ornato P...,Non d' altra forza d' animo fu ornato Ponzio A...
3,<2it> Se questo piace a tutti e se 'l tempo ha...,Se questo piace a tutti e se 'l tempo ha bisog...
4,<2it> Officio di questa arte pare che sia dice...,Officio di questa arte sembra che sia dire app...
5,<2it> Ecco e larghi ventipiovoli caggiono dell...,Ecco e larghi ventipiovoli caggiono delle riso...
6,<2it> Però che or chi spererebbe quello che ez...,Ma chi sperarebbe che questi che non vogliono ...
7,<2it> I vendimenti de' morti et le presure de'...,I vendimenti de' morti et le presure de' vivi ...
8,"<2it> Acciocché quegli, il quale ora per le su...","Acciocché colui, il quale ora per le sue gran ..."
9,<2it> Gli uomini spessamente a stare fermi nel...,Gli uomini spesso a stare fermi nella bugia in...


# Saving the Results

We'll save the original sentences along with their translations to a new CSV file.

In [22]:
output_file = 'madlad/madlad_translations.csv'
df.to_csv(output_file, index=False)

print(f"Saved translations to {output_file}")

Saved translations to madlad/madlad_translations.csv


In [4]:
# Read the saved translations file
translations_df = pd.read_csv('madlad/madlad_translations.csv')

# Show the first few rows
print("\nFirst 5 rows:")
translations_df.head()


First 5 rows:


,Author,Date,Region,Sentence,Modern_Italian
0,Brunetto Latini,1260-61,fior.,<2it> quella guerra ben fatta l' opera perché ...,Et dall' altra parte Aiaces era un cavaliere f...
1,Bono Giamboni,1292,fior.,"<2it> crudele, e di tutte le colpe pigli vende...","E di tutte le colpe pigli vendetta, come dice ..."
2,Valerio Massimo (red. V1,1336,fior.,<2it> Non d' altra forza d' animo fue ornato P...,Non d' altra forza d' animo fu ornato Ponzio A...
3,Lucano volg. (ed. Marinoni),1330/40,prat.,<2it> Se questo piace a tutti e se 'l tempo ha...,Se questo piace a tutti e se 'l tempo ha bisog...
4,Brunetto Latini,1260-61,fior.,<2it> Officio di questa arte pare che sia dice...,Officio di questa arte sembra che sia dire app...


In [5]:
import os
from dotenv import load_dotenv
from google import genai
import time

load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")
client = genai.Client(api_key=api_key)

def evaluate_translation(row, without_context = False) -> int:
    criteria = (
        "1. Completely unacceptable translation: the translation has no pertinence with the original meaning, the generated sentence is either gibberish or something that makes no sense.\n"
        "2. Severe semantic errors, omissions or substantial add ons on the original sentence. The errors are of semantic and syntactic nature. It’s still something no human would ever write.\n"
        "3. Partially wrong translation, the translation is lackluster, it contains errors, but are mostly minor errors, like typos, or small semantic errors.\n"
        "4. Good translation. The translation is mostly right, substantially faithful to the original text, but the style does not perfectly match the original sentence, still fluent and comprehensible, and could semantically acceptable.\n"
        "5. Perfect translation. The translation is accurate, fluent, complete and coherent. It retained the original meaning as much as it could."
    )

    if without_context:
        # Prompt without context
        prompt = (
            f"Rate the following translation on a scale of 1-5 based on these criteria:\n"
            f"{criteria}\n\n"
            f"Original sentence:\n\"{row['Sentence']}\"\n\n"
            f"Translated sentence:\n\"{row['Modern_Italian']}\"\n\n"
            f"Provide only the rating (1-5)."
        )
    else:
        # Prompt with context
        prompt = (
            f"Rate this translation on a scale of 1-5 based on these criteria:\n"
            f"{criteria}\n\n"
            "Context:\n"
            f" • Author: {row['Author']}\n"
            f" • Date: {row['Date']}\n"
            f" • Region: {row['Region']}\n\n"
            f"Original sentence:\n\"{row['Sentence']}\"\n\n"
            f"Translated into Modern Italian:\n\"{row['Modern_Italian']}\"\n\n"
            "Provide only the rating (1-5)."
        )
    resp = client.models.generate_content(
        model="gemini-2.0-flash",
        contents=prompt
    )
    try:
        return int(resp.text.strip())
    except ValueError:
        return None

In [8]:
RATE_LIMIT_SECONDS = 4.5

ratings = []
for _, row in tqdm(translations_df.iterrows(), total=len(translations_df), desc="Evaluating translations"):
    rating = evaluate_translation(row)
    ratings.append(rating)
    time.sleep(RATE_LIMIT_SECONDS)

translations_df['gemini_eval'] = ratings

Evaluating translations: 100%|██████████| 97/97 [07:54<00:00,  4.90s/it]


In [12]:
# Now evaluate translations without context
RATE_LIMIT_SECONDS = 4.5

ratings_no_context = []
for _, row in tqdm(translations_df.iterrows(), total=len(translations_df), desc="Evaluating without context"):
    rating = evaluate_translation(row, without_context=True)
    ratings_no_context.append(rating)
    time.sleep(RATE_LIMIT_SECONDS)

translations_df['gemini_eval_no_context'] = ratings_no_context

# Print summary statistics
print("\nEvaluation summary:")
print(f"Average rating with context: {translations_df['gemini_eval'].mean():.2f}")
print(f"Average rating without context: {translations_df['gemini_eval_no_context'].mean():.2f}")
print(f"Difference: {(translations_df['gemini_eval'] - translations_df['gemini_eval_no_context']).mean():.2f}")

translations_df.to_csv('madlad/madlad_translations_with_eval.csv', index=False)

Evaluating without context: 100%|██████████| 97/97 [07:57<00:00,  4.92s/it]


Evaluation summary:
Average rating with context: 3.06
Average rating without context: 2.80
Difference: 0.26
